In [ ]:
# this file was executed in kaggle environment, it is not intended to be run on local machine. It is here for reference only.

In [ ]:
import os, gc, shutil
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device        :", device)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))
    print("GPU RAM       :", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

In [ ]:
DATA_PATH = '/kaggle/input/datasets/sarahdje/hotel-reviews-nlp'

train_df = pd.read_csv(f'{DATA_PATH}/train.csv').dropna(subset=['text']).reset_index(drop=True)
val_df   = pd.read_csv(f'{DATA_PATH}/val.csv').dropna(subset=['text']).reset_index(drop=True)
test_df  = pd.read_csv(f'{DATA_PATH}/test.csv').dropna(subset=['text']).reset_index(drop=True)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print("Columns:", list(train_df.columns))

In [ ]:
label2id = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
id2label  = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

for df in [train_df, val_df, test_df]:
    df['label_id'] = df['label'].map(label2id)

train_sample, _ = train_test_split(
    train_df,
    train_size=200_000,
    stratify=train_df['label_id'],
    random_state=42
)
train_sample = train_sample.reset_index(drop=True)

print(f"Training sample: {len(train_sample):,} rows")
counts = train_sample['label_id'].value_counts().sort_index()
for lid, count in counts.items():
    print(f"  {id2label[lid]:<12} {count:,}  ({count/len(train_sample)*100:.1f}%)")

In [ ]:
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_sample['label_id'].values
)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

print("Class weights:")
for label, w in zip(['Negative','Neutral','Positive'], weights):
    print(f"  {label:<12} {w:.4f}")

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 128

tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

class HotelReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts      = texts.tolist()
        self.labels     = labels.tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'      : encoding['input_ids'].squeeze(0),
            'attention_mask' : encoding['attention_mask'].squeeze(0),
            'labels'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = HotelReviewDataset(train_sample['text'], train_sample['label_id'], tokenizer)
val_dataset   = HotelReviewDataset(val_df['text'],       val_df['label_id'],       tokenizer)
test_dataset  = HotelReviewDataset(test_df['text'],      test_df['label_id'],      tokenizer)

print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}")

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Batches per epoch: {len(train_loader):,}")

In [ ]:
EPOCHS       = 3
TRAIN_SIZE   = 200_000
total_steps  = (TRAIN_SIZE // BATCH_SIZE) * EPOCHS
warmup_steps = total_steps // 10

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
loss_fn   = CrossEntropyLoss(weight=class_weights)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

gc.collect()
torch.cuda.empty_cache()

print(f"Model ready on {device}")
print(f"Total steps : {total_steps:,} | Warmup: {warmup_steps:,}")

In [ ]:
from IPython.display import FileLink, display
import ipywidgets as widgets

In [ ]:
from IPython.display import FileLink, display
import shutil, os, gc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)

CKPT_DIR   = '/kaggle/working/checkpoints'
OUTPUT_DIR = '/kaggle/working'
os.makedirs(CKPT_DIR, exist_ok=True)

history     = {'train_loss': [], 'val_loss': [], 'val_f1': []}
best_f1     = 0
best_epoch  = 0
label_names = ['Negative', 'Neutral', 'Positive']

for epoch in range(EPOCHS):

    # ── TRAIN ──────────────────────────────────────────────
    model.train()
    total_train_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

        if (batch_idx + 1) % 200 == 0:
            avg = total_train_loss / (batch_idx + 1)
            print(f"  Epoch {epoch+1} | Batch {batch_idx+1:,}/{len(train_loader):,} | Loss: {avg:.4f}")

    avg_train_loss = total_train_loss / len(train_loader)

    # ── VALIDATE ───────────────────────────────────────────
    model.eval()
    total_val_loss  = 0
    all_preds       = []
    all_labels_list = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = loss_fn(outputs.logits, labels)
            total_val_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels_list.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)
    val_f1       = f1_score(all_labels_list, all_preds, average='macro')

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_f1'].append(val_f1)

    print(f"\nEpoch {epoch+1}/{EPOCHS} complete")
    print(f"  Train Loss    : {avg_train_loss:.4f}")
    print(f"  Val Loss      : {avg_val_loss:.4f}")
    print(f"  Val F1 (macro): {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1    = val_f1
        best_epoch = epoch + 1

    # ── SAVE + ZIP + AUTO-DOWNLOAD after every epoch ───────
    ckpt_path = f'{CKPT_DIR}/epoch_{epoch+1}'
    os.makedirs(ckpt_path, exist_ok=True)
    model.save_pretrained(ckpt_path)
    tokenizer.save_pretrained(ckpt_path)

    zip_path = f'{OUTPUT_DIR}/model_epoch_{epoch+1}'
    shutil.make_archive(zip_path, 'zip', ckpt_path)
    print(f"  Model zipped → {zip_path}.zip")

    # Auto-download triggers immediately in the browser
    from kaggle_secrets import UserSecretsClient
    display(FileLink(f'model_epoch_{epoch+1}.zip'))
    print(f"  ^^^ Click above to download epoch {epoch+1} model ^^^")

    gc.collect()
    torch.cuda.empty_cache()
    print()

# ── TEST EVALUATION ────────────────────────────────────────
print("Running test evaluation...")
model.eval()
all_preds       = []
all_labels_list = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels_list.extend(labels.cpu().numpy())

print("="*50)
print("TEST SET RESULTS")
print("="*50)
print(f"Accuracy     : {accuracy_score(all_labels_list, all_preds):.4f}")
print(f"F1 (macro)   : {f1_score(all_labels_list, all_preds, average='macro'):.4f}")
print(f"F1 (weighted): {f1_score(all_labels_list, all_preds, average='weighted'):.4f}")
print()
print(classification_report(all_labels_list, all_preds, target_names=label_names))

# ── CONFUSION MATRIX ───────────────────────────────────────
cm = confusion_matrix(all_labels_list, all_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('DistilBERT — Confusion Matrix (Test Set)')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix_distilbert_2.png', dpi=150)
plt.show()
display(FileLink('confusion_matrix_distilbert_2.png'))

# ── TRAINING CURVE ─────────────────────────────────────────
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_range, history['train_loss'], 'o-', label='Train loss')
axes[0].plot(epochs_range, history['val_loss'],   'o-', label='Val loss')
axes[0].set_title('Loss per epoch')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[1].plot(epochs_range, history['val_f1'], 'o-', color='green')
axes[1].set_title('Val F1 (macro) per epoch')
axes[1].set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curve_distilbert_2.png', dpi=150)
plt.show()
display(FileLink('training_curve_distilbert_2.png'))

# ── BEST MODEL FINAL ZIP ───────────────────────────────────
best_ckpt = f'{CKPT_DIR}/epoch_{best_epoch}'
shutil.make_archive(f'{OUTPUT_DIR}/distilbert_best_final_2', 'zip', best_ckpt)
print(f"\nBest model: epoch {best_epoch} — Val F1: {best_f1:.4f}")
display(FileLink('distilbert_best_final_2.zip'))
print("^^^ Download this — it's your final model ^^^")